# 🔗 HPVD × Knowledge Layer — Integration Demo

Notebook ini mendemonstrasikan alur **end-to-end** HPVD mengambil data dari Knowledge Layer (KL) yang live, 
menjalankan pipeline retrieval, dan menghasilkan **J14 (RetrievalRaw)**, **J15 (PhaseFilteredSet)**, dan **J16 (AnalogFamilyAssignment)**.

---

**Pipeline:**
```
KL API (live) → KLDocumentLoader → DocumentChunk[] → HPVD Pipeline → J14 → J15 → J16
```

## 1. Connect to Knowledge Layer

In [1]:
from src.hpvd.adapters.kl_client import KLClient

KL_URL = "https://knowledge-layer-production.up.railway.app"
client = KLClient(base_url=KL_URL)

# Health check
health = client.health_check()
print(f"✅ KL API reachable: {health}")

✅ KL API reachable: {'message': 'Knowledge Layer API'}


## 2. Fetch Documents from KL

In [2]:
TENANT_ID = "BANKING_CORE"

documents = client.list_documents(TENANT_ID, limit=100)
print(f"📄 Total documents in KL for tenant '{TENANT_ID}': {len(documents)}")
print()

# Show summary by document type
from collections import Counter
type_counts = Counter(d.document_type for d in documents)
print("Document types:")
for doc_type, count in type_counts.most_common():
    print(f"  {doc_type:25s} → {count} docs")

📄 Total documents in KL for tenant 'BANKING_CORE': 36

Document types:
  NO_PREJUDICE              → 4 docs
  ESITO_LETTER              → 4 docs
  OTHER                     → 3 docs
  DELIBERA                  → 3 docs
  INTIMAZIONE               → 2 docs
  CREDIT_REPORT             → 2 docs
  NO_DEFAULT                → 2 docs
  PEC_RECEIPT               → 2 docs
  CREDIT_SPECIFICATION      → 2 docs
  IDENTITY_DOC              → 2 docs
  CONTRACT                  → 2 docs
  EROGAZIONE                → 2 docs
  AMMORTAMENTO              → 2 docs
  RATE_DECLARATION          → 2 docs
  COMPLIANCE                → 1 docs
  RISK_DECLARATION          → 1 docs


In [3]:
# Show first 10 documents as a table
import pandas as pd

df_docs = pd.DataFrame([
    {
        "id": d.id[:8] + "...",
        "title": d.title[:60],
        "type": d.document_type,
        "created_at": d.created_at[:19] if d.created_at else "",
    }
    for d in documents[:10]
])
df_docs

,id,title,type,created_at
0,715f9a24...,[1437613] 1. intimazione_17-11-2025 12.47.20.pdf,INTIMAZIONE,2026-03-05T05:49:36
1,d3befefe...,[1437613] 10. no concordato_17-11-2025 12.50.0...,COMPLIANCE,2026-03-05T05:49:36
2,07d1f8f2...,[1437613] 11. CR 03 2020_17-11-2025 12.50.10.pdf,CREDIT_REPORT,2026-03-05T05:49:36
3,7e95c03a...,[1437613] 11. assenza inadempienze_17-11-2025 ...,NO_DEFAULT,2026-03-05T05:49:36
4,2f2f583b...,[1437613] 12. assenza pregiudizievoli_17-11-20...,NO_PREJUDICE,2026-03-05T05:49:36
5,4fe79dc5...,[1437613] 1437613_23-05-2020 10.41.37.pdf,OTHER,2026-03-05T05:49:36
6,f709525f...,[1437613] 2. consegna pec_17-11-2025 12.47.27.pdf,PEC_RECEIPT,2026-03-05T05:49:36
7,f26c7a6c...,[1437613] 3. precisazione del credito_17-11-20...,CREDIT_SPECIFICATION,2026-03-05T05:49:36
8,3f3d2a0d...,[1437613] 4. allegato 4 bis_17-11-2025 12.47.4...,IDENTITY_DOC,2026-03-05T05:49:36
9,529dad2a...,[1437613] 4. patente e cf guidozzi andrea_17-1...,IDENTITY_DOC,2026-03-05T05:49:36


## 3. Load & Convert to HPVD DocumentChunks

In [4]:
from src.hpvd.adapters.kl_document_loader import KLDocumentLoader

loader = KLDocumentLoader(client)
chunks = loader.load_as_chunks(TENANT_ID)

print(f"🧩 Converted {len(chunks)} documents → {len(chunks)} DocumentChunks")
print()

# Show topic distribution
topic_counts = Counter(c.topic for c in chunks)
print("Topic distribution (mapped from document_type):")
for topic, count in topic_counts.most_common():
    print(f"  {topic:25s} → {count} chunks")

🧩 Converted 36 documents → 36 DocumentChunks

Topic distribution (mapped from document_type):
  compliance                → 7 chunks
  legal_notice              → 4 chunks
  credit_analysis           → 4 chunks
  outcome                   → 4 chunks
  other                     → 3 chunks
  bank_decision             → 3 chunks
  risk_assessment           → 3 chunks
  identity                  → 2 chunks
  contract                  → 2 chunks
  disbursement              → 2 chunks
  repayment_plan            → 2 chunks


In [5]:
# Show sample chunks
df_chunks = pd.DataFrame([
    {
        "chunk_id": c.chunk_id[:15] + "...",
        "topic": c.topic,
        "doc_type": c.doc_type,
        "text_preview": c.text[:80],
    }
    for c in chunks[:10]
])
df_chunks

,chunk_id,topic,doc_type,text_preview
0,kl_715f9a24-18b...,legal_notice,INTIMAZIONE,[1437613] 1. intimazione_17-11-2025 12.47.20.p...
1,kl_d3befefe-99e...,compliance,COMPLIANCE,[1437613] 10. no concordato_17-11-2025 12.50.0...
2,kl_07d1f8f2-802...,credit_analysis,CREDIT_REPORT,[1437613] 11. CR 03 2020_17-11-2025 12.50.10.p...
3,kl_7e95c03a-a59...,compliance,NO_DEFAULT,[1437613] 11. assenza inadempienze_17-11-2025 ...
4,kl_2f2f583b-1d4...,compliance,NO_PREJUDICE,[1437613] 12. assenza pregiudizievoli_17-11-20...
5,kl_4fe79dc5-889...,other,OTHER,[1437613] 1437613_23-05-2020 10.41.37.pdf. Doc...
6,kl_f709525f-22c...,legal_notice,PEC_RECEIPT,[1437613] 2. consegna pec_17-11-2025 12.47.27....
7,kl_f26c7a6c-2c8...,credit_analysis,CREDIT_SPECIFICATION,[1437613] 3. precisazione del credito_17-11-20...
8,kl_3f3d2a0d-c7c...,identity,IDENTITY_DOC,[1437613] 4. allegato 4 bis_17-11-2025 12.47.4...
9,kl_529dad2a-b79...,identity,IDENTITY_DOC,[1437613] 4. patente e cf guidozzi andrea_17-1...


## 4. Build HPVD Index & Run Pipeline

Kita akan menjalankan pipeline HPVD lengkap:
1. **Build index** dari DocumentChunks (embedding + FAISS)
2. **Construct J13** (PostCoreQuery) — simulasi query dari Serving Adapter
3. **Run pipeline** → produce J14, J15, J16

In [6]:
from src.hpvd.adapters.strategies.document_strategy import (
    DocumentRetrievalStrategy, 
    DocumentRetrievalConfig,
)
from src.hpvd.adapters.pipeline_engine import HPVDPipelineEngine

# Build index
strategy = DocumentRetrievalStrategy(
    DocumentRetrievalConfig(min_similarity=0.0)
)
strategy.build_index(chunks)
print(f"✅ FAISS index built with {len(chunks)} vectors")

# Create pipeline
pipeline = HPVDPipelineEngine(strategies=[strategy])
print(f"✅ Pipeline ready (registered domains: document)")

d:\project\M22\HPVD-M22\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1047.93it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS index built with 36 vectors
✅ Pipeline ready (registered domains: document)


### 4.1 Construct J13 — PostCoreQuery

Ini adalah input query yang dalam pipeline penuh akan datang dari **Serving Adapter (Stage 11)**.

In [7]:
import json

j13_query = {
    "query_id": "Q_LOAN_SUPPORT_001",
    "scope": {
        "domain": "banking",
        "action_class": "LOAN_EXECUTION",
    },
    "allowed_topics": [],  # no topic filter = search all
    "allowed_doc_types": [],
    "query_payload": {
        "text": "credit risk assessment loan default payment",
    },
}

print("📥 J13 — PostCoreQuery:")
print(json.dumps(j13_query, indent=2))

📥 J13 — PostCoreQuery:
{
  "query_id": "Q_LOAN_SUPPORT_001",
  "scope": {
    "domain": "banking",
    "action_class": "LOAN_EXECUTION"
  },
  "allowed_topics": [],
  "allowed_doc_types": [],
  "query_payload": {
    "text": "credit risk assessment loan default payment"
  }
}


### 4.2 Execute Pipeline → J14, J15, J16

In [8]:
# Run the pipeline
output = pipeline.process_query(j13_query, k=10)

print(f"✅ Pipeline complete!")
print(f"   J14 candidates : {len(output.j14.candidates)}")
print(f"   J15 accepted   : {len(output.j15.accepted)}")
print(f"   J15 rejected   : {len(output.j15.rejected)}")
print(f"   J16 families   : {output.j16.total_families}")
print(f"   J16 members    : {output.j16.total_members}")

✅ Pipeline complete!
   J14 candidates : 10
   J15 accepted   : 10
   J15 rejected   : 0
   J16 families   : 6
   J16 members    : 10


## 5. Inspect J14 — HPVD Retrieval Raw

Hasil retrieval mentah: daftar kandidat analog dengan **calibrated similarity score**.

In [9]:
print("📤 J14 — HPVD_RetrievalRaw")
print(f"   schema_id : {output.j14.schema_id}")
print(f"   query_id  : {output.j14.query_id}")
print(f"   domain    : {output.j14.domain}")
print(f"   candidates: {len(output.j14.candidates)}")
print()

# Show candidates as table
df_j14 = pd.DataFrame([
    {
        "rank": i + 1,
        "candidate_id": c["candidate_id"][:20] + "...",
        "score": round(c["score"], 4),
        "topic": c.get("metadata", {}).get("topic", ""),
        "doc_type": c.get("metadata", {}).get("doc_type", ""),
        "text_preview": c.get("metadata", {}).get("text_preview", "")[:50],
    }
    for i, c in enumerate(output.j14.candidates)
])
df_j14

📤 J14 — HPVD_RetrievalRaw
   schema_id : manithy.hpvd_retrieval_raw.v1
   query_id  : Q_LOAN_SUPPORT_001
   domain    : document
   candidates: 10



,rank,candidate_id,score,topic,doc_type,text_preview
0,1,kl_07d1f8f2-802f-443...,0.3924,credit_analysis,CREDIT_REPORT,[1437613] 11. CR 03 2020_17-11-2025 12.50.10.p...
1,2,kl_f26c7a6c-2c87-4d1...,0.3748,credit_analysis,CREDIT_SPECIFICATION,[1437613] 3. precisazione del credito_17-11-20...
2,3,kl_75b4af9a-9290-401...,0.3631,credit_analysis,CREDIT_SPECIFICATION,[1440632] 3. Precisazione del credito_18-12-20...
3,4,kl_87e5ee60-50fd-40b...,0.3364,risk_assessment,RISK_DECLARATION,[1437613] 9. Dichiarazione Rischio di Credito_...
4,5,kl_54d09cea-c6b3-4b8...,0.3335,credit_analysis,CREDIT_REPORT,[1440632] 11. Prima Info CR MARZO 2020_18-12-2...
5,6,kl_7ceae5a8-f873-421...,0.1923,contract,CONTRACT,[1440632] 6. Contratto finanziamento_18-12-202...
6,7,kl_f709525f-22cf-4d6...,0.1877,legal_notice,PEC_RECEIPT,[1437613] 2. consegna pec_17-11-2025 12.47.27....
7,8,kl_d3befefe-99ea-48c...,0.1789,compliance,COMPLIANCE,[1437613] 10. no concordato_17-11-2025 12.50.0...
8,9,kl_527228cc-367d-4c1...,0.1625,contract,CONTRACT,[1437613] 6. contratto e pda_11zon (2)_17-11-2...
9,10,kl_3f3d2a0d-c7ca-4c0...,0.1435,identity,IDENTITY_DOC,[1437613] 4. allegato 4 bis_17-11-2025 12.47.4...


In [10]:
# Full J14 JSON
print(json.dumps(output.j14.to_dict(), indent=2, ensure_ascii=False))

{
  "schema_id": "manithy.hpvd_retrieval_raw.v1",
  "query_id": "Q_LOAN_SUPPORT_001",
  "domain": "document",
  "candidates": [
    {
      "candidate_id": "kl_07d1f8f2-802f-443f-b861-5735d716c385",
      "score": 0.3923553228378296,
      "metadata": {
        "topic": "credit_analysis",
        "doc_type": "CREDIT_REPORT",
        "text_preview": "[1437613] 11. CR 03 2020_17-11-2025 12.50.10.pdf. Document type: CREDIT_REPORT. Created by: seed_script_v1"
      },
      "source_domain": "document"
    },
    {
      "candidate_id": "kl_f26c7a6c-2c87-4d1d-88be-ae22e87c5ebc",
      "score": 0.3748058080673218,
      "metadata": {
        "topic": "credit_analysis",
        "doc_type": "CREDIT_SPECIFICATION",
        "text_preview": "[1437613] 3. precisazione del credito_17-11-2025 12.47.35.pdf. Document type: CREDIT_SPECIFICATION. Created by: seed_scr"
      },
      "source_domain": "document"
    },
    {
      "candidate_id": "kl_75b4af9a-9290-4015-838a-43f5c42f2202",
      "score": 0

## 6. Inspect J15 — Phase Filtered Set

Kandidat yang sudah difilter berdasarkan **phase/topic consistency**.

In [11]:
print("📤 J15 — PhaseFilteredSet")
print(f"   schema_id      : {output.j15.schema_id}")
print(f"   accepted count : {len(output.j15.accepted)}")
print(f"   rejected count : {len(output.j15.rejected)}")
print(f"   filter_criteria: {output.j15.filter_criteria}")
print()

# Show accepted candidates
if output.j15.accepted:
    df_j15 = pd.DataFrame([
        {
            "candidate_id": c["candidate_id"][:20] + "...",
            "score": round(c["score"], 4),
            "status": "ACCEPTED",
        }
        for c in output.j15.accepted
    ])
    display(df_j15)

if output.j15.rejected:
    print(f"\nRejected ({len(output.j15.rejected)} candidates):")
    for r in output.j15.rejected:
        print(f"  {r['candidate_id'][:20]}... — reason: {r.get('reason', 'N/A')}")

📤 J15 — PhaseFilteredSet
   schema_id      : manithy.phase_filtered_set.v1
   accepted count : 10
   rejected count : 0
   filter_criteria: {'allowed_topics': [], 'allowed_doc_types': []}



,candidate_id,score,status
0,kl_07d1f8f2-802f-443...,0.3924,ACCEPTED
1,kl_f26c7a6c-2c87-4d1...,0.3748,ACCEPTED
2,kl_75b4af9a-9290-401...,0.3631,ACCEPTED
3,kl_87e5ee60-50fd-40b...,0.3364,ACCEPTED
4,kl_54d09cea-c6b3-4b8...,0.3335,ACCEPTED
5,kl_7ceae5a8-f873-421...,0.1923,ACCEPTED
6,kl_f709525f-22cf-4d6...,0.1877,ACCEPTED
7,kl_d3befefe-99ea-48c...,0.1789,ACCEPTED
8,kl_527228cc-367d-4c1...,0.1625,ACCEPTED
9,kl_3f3d2a0d-c7ca-4c0...,0.1435,ACCEPTED


In [12]:
# Full J15 JSON
print(json.dumps(output.j15.to_dict(), indent=2, ensure_ascii=False))

{
  "schema_id": "manithy.phase_filtered_set.v1",
  "query_id": "Q_LOAN_SUPPORT_001",
  "accepted": [
    {
      "candidate_id": "kl_07d1f8f2-802f-443f-b861-5735d716c385",
      "score": 0.3923553228378296,
      "metadata": {
        "topic": "credit_analysis",
        "doc_type": "CREDIT_REPORT",
        "text_preview": "[1437613] 11. CR 03 2020_17-11-2025 12.50.10.pdf. Document type: CREDIT_REPORT. Created by: seed_script_v1"
      },
      "source_domain": "document"
    },
    {
      "candidate_id": "kl_f26c7a6c-2c87-4d1d-88be-ae22e87c5ebc",
      "score": 0.3748058080673218,
      "metadata": {
        "topic": "credit_analysis",
        "doc_type": "CREDIT_SPECIFICATION",
        "text_preview": "[1437613] 3. precisazione del credito_17-11-2025 12.47.35.pdf. Document type: CREDIT_SPECIFICATION. Created by: seed_scr"
      },
      "source_domain": "document"
    },
    {
      "candidate_id": "kl_75b4af9a-9290-4015-838a-43f5c42f2202",
      "score": 0.36310291290283203,
      

## 7. Inspect J16 — Analog Family Assignment

Kandidat yang sudah dikelompokkan menjadi **analog families** dengan:
- `coherence` — seberapa koheren family tersebut
- `structural_signature` — ciri struktural family
- `uncertainty_flags` — apakah ada ketidakpastian

In [13]:
print("📤 J16 — AnalogFamilyAssignment")
print(f"   schema_id     : {output.j16.schema_id}")
print(f"   total_families: {output.j16.total_families}")
print(f"   total_members : {output.j16.total_members}")
print()

for fam in output.j16.families:
    print(f"{'='*60}")
    print(f"Family: {fam['family_id']}")
    print(f"  Phase/Topic    : {fam['structural_signature']['phase']}")
    print(f"  Members        : {fam['coherence']['size']}")
    print(f"  Mean confidence: {fam['coherence']['mean_confidence']:.4f}")
    print(f"  Dispersion     : {fam['coherence']['dispersion']:.4f}")
    print(f"  Weak support   : {fam['uncertainty_flags']['weak_support']}")
    print(f"  Phase boundary : {fam['uncertainty_flags']['phase_boundary']}")
    print(f"  Members:")
    for m in fam['members']:
        print(f"    - {m['candidate_id'][:25]}...  score={m['score']:.4f}")

📤 J16 — AnalogFamilyAssignment
   schema_id     : manithy.analog_family_assignment.v1
   total_families: 6
   total_members : 10

Family: DF_001
  Phase/Topic    : compliance
  Members        : 1
  Mean confidence: 0.1789
  Dispersion     : 0.0000
  Weak support   : True
  Phase boundary : False
  Members:
    - kl_d3befefe-99ea-48ca-80c...  score=0.1789
Family: DF_002
  Phase/Topic    : contract
  Members        : 2
  Mean confidence: 0.1774
  Dispersion     : 0.0149
  Weak support   : True
  Phase boundary : False
  Members:
    - kl_7ceae5a8-f873-4214-a93...  score=0.1923
    - kl_527228cc-367d-4c19-a94...  score=0.1625
Family: DF_003
  Phase/Topic    : credit_analysis
  Members        : 4
  Mean confidence: 0.3659
  Dispersion     : 0.0214
  Weak support   : True
  Phase boundary : False
  Members:
    - kl_07d1f8f2-802f-443f-b86...  score=0.3924
    - kl_f26c7a6c-2c87-4d1d-88b...  score=0.3748
    - kl_75b4af9a-9290-4015-838...  score=0.3631
    - kl_54d09cea-c6b3-4b89-9df...  sco

In [14]:
# Family summary table
df_j16 = pd.DataFrame([
    {
        "family_id": f["family_id"],
        "topic": f["structural_signature"]["phase"],
        "members": f["coherence"]["size"],
        "mean_confidence": round(f["coherence"]["mean_confidence"], 4),
        "dispersion": round(f["coherence"]["dispersion"], 4),
        "weak_support": f["uncertainty_flags"]["weak_support"],
    }
    for f in output.j16.families
])
df_j16

,family_id,topic,members,mean_confidence,dispersion,weak_support
0,DF_001,compliance,1,0.1789,0.0000,True
1,DF_002,contract,2,0.1774,0.0149,True
2,DF_003,credit_analysis,4,0.3659,0.0214,True
3,DF_004,identity,1,0.1435,0.0000,True
4,DF_005,legal_notice,1,0.1877,0.0000,True
5,DF_006,risk_assessment,1,0.3364,0.0000,True


In [15]:
# Full J16 JSON
print(json.dumps(output.j16.to_dict(), indent=2, ensure_ascii=False))

{
  "schema_id": "manithy.analog_family_assignment.v1",
  "query_id": "Q_LOAN_SUPPORT_001",
  "families": [
    {
      "family_id": "DF_001",
      "members": [
        {
          "candidate_id": "kl_d3befefe-99ea-48ca-80c3-4641a49f6019",
          "score": 0.17885693907737732,
          "metadata": {
            "topic": "compliance",
            "doc_type": "COMPLIANCE",
            "text_preview": "[1437613] 10. no concordato_17-11-2025 12.50.00.pdf. Document type: COMPLIANCE. Created by: seed_script_v1"
          },
          "source_domain": "document"
        }
      ],
      "coherence": {
        "mean_confidence": 0.17885693907737732,
        "dispersion": 0.0,
        "size": 1
      },
      "structural_signature": {
        "phase": "compliance",
        "avg_K": null,
        "avg_LTV": null,
        "avg_LVC": null
      },
      "uncertainty_flags": {
        "phase_boundary": false,
        "weak_support": true,
        "partial_overlap": false
      }
    },
    {
  

## 8. Save Results to File

Simpan output lengkap ke JSON file untuk referensi.

In [16]:
import os
from datetime import datetime

# Save full pipeline output
output_dir = "hpvd_outputs/kl_integration"
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = os.path.join(output_dir, f"pipeline_output_{timestamp}.json")

full_output = {
    "metadata": {
        "timestamp": timestamp,
        "kl_url": KL_URL,
        "tenant_id": TENANT_ID,
        "total_kl_documents": len(documents),
        "total_chunks": len(chunks),
    },
    "j13_query": j13_query,
    "pipeline_output": output.to_dict(),
}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(full_output, f, indent=2, ensure_ascii=False)

print(f"💾 Output saved to: {output_file}")
print(f"   File size: {os.path.getsize(output_file):,} bytes")

💾 Output saved to: hpvd_outputs/kl_integration\pipeline_output_20260310_234449.json
   File size: 18,382 bytes


## 9. Try Different Queries

Coba ubah query di bawah untuk lihat bagaimana HPVD meresponse query yang berbeda.

In [17]:
# Try different queries!
queries = [
    {"text": "bank deliberation approval decision", "label": "Bank Decision"},
    {"text": "legal notice payment intimation", "label": "Legal Notice"},
    {"text": "fidejussione garanzia guarantee", "label": "Guarantee"},
    {"text": "loan contract finanziamento", "label": "Contract"},
]

results_summary = []
for q in queries:
    j13 = {
        "query_id": f"multi_q_{q['label'].replace(' ', '_').lower()}",
        "scope": {"domain": "banking"},
        "allowed_topics": [],
        "query_payload": {"text": q["text"]},
    }
    out = pipeline.process_query(j13, k=5)
    
    top_candidate = out.j14.candidates[0] if out.j14.candidates else {}
    results_summary.append({
        "query": q["label"],
        "candidates": len(out.j14.candidates),
        "families": out.j16.total_families,
        "top_score": round(top_candidate.get("score", 0), 4),
        "top_topic": top_candidate.get("metadata", {}).get("topic", ""),
    })

df_results = pd.DataFrame(results_summary)
df_results

,query,candidates,families,top_score,top_topic
0,Bank Decision,5,3,0.3136,credit_analysis
1,Legal Notice,5,3,0.3301,credit_analysis
2,Guarantee,5,5,0.3211,compliance
3,Contract,5,2,0.5778,contract


## 10. Topic-Filtered Query

Coba query dengan `allowed_topics` untuk membatasi retrieval ke topic tertentu.

In [18]:
# Query with topic filter
j13_filtered = {
    "query_id": "Q_FILTERED_CREDIT",
    "scope": {"domain": "banking"},
    "allowed_topics": ["credit_analysis"],  # Only credit-related docs
    "query_payload": {"text": "credit risk assessment"},
}

out_filtered = pipeline.process_query(j13_filtered, k=10)

print(f"🔍 Filtered query (topic=credit_analysis):")
print(f"   Candidates: {len(out_filtered.j14.candidates)}")
print(f"   Families  : {out_filtered.j16.total_families}")
print()

for c in out_filtered.j14.candidates:
    print(f"  score={c['score']:.4f}  topic={c['metadata']['topic']}  {c['metadata'].get('text_preview', '')[:50]}")

🔍 Filtered query (topic=credit_analysis):
   Candidates: 4
   Families  : 1

  score=0.5226  topic=credit_analysis  [1437613] 11. CR 03 2020_17-11-2025 12.50.10.pdf. 
  score=0.5175  topic=credit_analysis  [1437613] 3. precisazione del credito_17-11-2025 1
  score=0.4906  topic=credit_analysis  [1440632] 3. Precisazione del credito_18-12-2025 1
  score=0.4371  topic=credit_analysis  [1440632] 11. Prima Info CR MARZO 2020_18-12-2025 


---

## Summary

| Stage | Output | Description |
|-------|--------|-------------|
| KL API | Documents | 36 banking PDFs from 6 cases |
| Loader | DocumentChunks | Converted with topic mapping |
| J14 | RetrievalRaw | Ranked candidates with similarity scores |
| J15 | PhaseFilteredSet | Accepted/rejected by phase filter |
| J16 | AnalogFamilyAssignment | Grouped into families with coherence metrics |